# W2 — From Exploratory Analysis to Credible Validation
### Line B (GRD): severity and insurance type in Chile's public hospital network

**Group 03 — Miguel Figueroa, Sebastián Bunzli**

**Business question:** Among privately-insured patients (Isapre / Particular)
who end up treated in Chile's public hospital network, is their clinical
severity profile different from FONASA beneficiaries?

**Unit of observation:** one GRD hospital discharge (egreso). Not one patient;
the same patient can appear several times (C1 §2.5).

**Intended decision:** help a *Servicio de Salud* or hospital management
analyst decide whether Isapre / Particular discharges should be monitored
as a distinct group or treated like all other discharges (C1 §1.1, §2.1).

**Future predictive use (not modelled in W2):** if extended to prediction,
the target would be `IR_29301_SEVERIDAD == "3"` (high severity), using only
features available at admission. The leakage audit (§7) and the chronological
split (§8) formalise what C1 §6.2 sketched.

**Execution model:**
1. If `data/processed/*.csv.gz` already exist **and** contain the full column
   set **and** `patient_hash` has real values, they are reused.
2. Otherwise, `§1` rebuilds them from the raw GRD / Establecimientos files
   under `RAW_DATA_ROOT`.
3. The remaining sections assume the processed files exist.

**AI-use disclosure:** AI assistance was used to draft boilerplate code and
to review written sections. Every number, choice and interpretation was
verified by the team against the source files. No AI-generated figures are
reported.

In [1]:
import os, re, random, unicodedata, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", message="Parsing dates", category=UserWarning)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Force regeneration even if files look complete (set True once if in doubt)
FORCE_REGENERATE = False

# ---------------- Path resolution (robust to subfolders) ----------------
HERE = Path.cwd()
_candidates = [HERE, *HERE.parents]
REPO = next((c for c in _candidates if (c / "data").exists() or (c / "scripts").exists()), HERE)
REPO = Path(os.environ.get("W2_ROOT", REPO)).resolve()
OUT_DIR = REPO / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_ROOT = Path(os.environ.get("RAW_DATA_ROOT",
                               r"C:\Users\sebal\Downloads\BI_T-main")).resolve()
GRD_DIR  = RAW_ROOT / "GRD"
EST_DIR  = RAW_ROOT / "Establecimientos"

YEARS = [2020, 2021, 2022, 2023, 2024]

GRD_FILES = {
    2020: GRD_DIR / "GRD_PUBLICO_2020.txt",
    2021: GRD_DIR / "GRD_PUBLICO_2021.txt",
    2022: GRD_DIR / "GRD_PUBLICO_EXTERNO_2022.txt",
    2023: GRD_DIR / "GRD_PUBLICO_2023.txt",
    2024: GRD_DIR / "GRD_PUBLICO_2024.txt",
}
ENCODINGS = {2020: "latin1", 2021: "latin1", 2022: "utf-16", 2023: "utf-16", 2024: "latin1"}

ESTAB_FILES = {y: EST_DIR / str(y) / f"Base_Establecimientos_{y}.xlsx" for y in YEARS}

BASE_COLS = ["PREVISION", "IR_29301_PESO", "IR_29301_SEVERIDAD",
             "IR_29301_MORTALIDAD", "TIPO_INGRESO"]

# Per-year files must contain these columns. `patient_hash` must ALSO have
# real values (checked in _files_are_stale).
REQUIRED_COLS = {
    "patient_hash", "episode_id",
    "is_dup_5cols", "is_dup_full", "is_dup_patient",
    "EDAD_num", "insurance_group", "PESO_num", "year", "valid",
}

INSURANCE_MAP = {
    "FONASA INSTITUCIONAL - (MAI) A": "FONASA", "FONASA INSTITUCIONAL - (MAI) B": "FONASA",
    "FONASA INSTITUCIONAL - (MAI) C": "FONASA", "FONASA INSTITUCIONAL - (MAI) D": "FONASA",
    "FONASA LIBRE ELECCIÓN (FMLE_B)": "FONASA", "FONASA LIBRE ELECCIÓN (FMLE_C)": "FONASA",
    "FONASA LIBRE ELECCIÓN (FMLE_D)": "FONASA",
    "ISAPRE": "PRIVATE", "PARTICULAR": "PRIVATE",
    "DIPRECA": "MILITARY", "CAPREDENA": "MILITARY",
    "CAJA DE PREVISIÓN FFAA (SISA)": "MILITARY",
    "DESCONOCIDO": None,
}

def _files_are_stale():
    """True if any processed file is missing, incomplete, or has no real patient hashes."""
    for y in YEARS:
        p = OUT_DIR / f"grd_{y}_extended.csv.gz"
        if not p.exists():
            print(f"[{y}] processed file missing -> stale")
            return True
        try:
            cols = set(pd.read_csv(p, compression="gzip", nrows=0).columns)
        except Exception as exc:
            print(f"[{y}] cannot read header ({exc}) -> stale")
            return True
        missing = REQUIRED_COLS - cols
        if missing:
            print(f"[{y}] processed file lacks {sorted(missing)} -> stale")
            return True
        if "patient_hash" in cols:
            s = pd.read_csv(p, compression="gzip", usecols=["patient_hash"], low_memory=False)
            if s["patient_hash"].notna().sum() == 0:
                print(f"[{y}] patient_hash all null -> stale")
                return True
    return False

if _files_are_stale():
    print("Stale processed files detected -> §1 will regenerate.\n")
    FORCE_REGENERATE = True

print("REPO        :", REPO)
print("OUT_DIR     :", OUT_DIR)
print("RAW_ROOT    :", RAW_ROOT, "(exists:", RAW_ROOT.exists(), ")")
print("FORCE_REGEN :", FORCE_REGENERATE)

REPO        : C:\Users\sebal\Downloads\w2_grd
OUT_DIR     : C:\Users\sebal\Downloads\w2_grd\data\processed
RAW_ROOT    : C:\Users\sebal\Downloads\BI_T-main (exists: True )
FORCE_REGEN : False


## 1. Regeneration of processed extracts (idempotent, stale-aware)

Rebuilds the five `.csv.gz` files with three additional columns
(`ID_BENEFICIARIO`, `IR_29301_COD_GRD`, `FECHA_NACIMIENTO`) and joins the
Establecimientos table.

Key design decisions:
- **`patient_hash`** is a deterministic uint64 hash of `ID_BENEFICIARIO`.
  Rows without an ID receive a unique placeholder, so they never falsely
  group together. This lets us compute a truly global `patient_id` after
  the multi-year concat, avoiding cross-year leakage in the GroupKFold.
- **`episode_id`** = `year * 10_000_000 + row_id`, unique across years.
- **Establecimientos join** uses `left_on`/`right_on` to preserve `estab_ESTAB`.

In [2]:
def _norm(s):
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().strip().lower()

def _find(columns, targets):
    for c in columns:
        if _norm(c) in targets:
            return c
    return None

PATTERNS = {
    "GRD":        [r"^IR_29301_COD_GRD$", r"^COD.*GRD", r"^GRD$"],
    "SEXO":       [r"^SEXO$", r"^GENERO$", r"^SEXO_DESC$"],
    "ESTAB":      [r"^COD_HOSPITAL$", r"^COD_?ESTAB", r"^CODIGO_?HOSP"],
    "FECHA_ING":  [r"^FECHA_INGRESO$", r"^FEC_INGRESO$"],
    "FECHA_EGR":  [r"^FECHAALTA$", r"^FECHA_EGRESO$", r"^FEC_EGRESO$"],
    "FECHA_NAC":  [r"^FECHA_NACIMIENTO$", r"^FEC_NACIMIENTO$"],
    "PATIENT_ID": [r"^ID_BENEFICIARIO$", r"^ID_PACIENTE$"],
}

def _detect(header_cols):
    out = {}
    for key, pats in PATTERNS.items():
        for pat in pats:
            matches = [c for c in header_cols if re.match(pat, c, re.IGNORECASE)]
            if matches:
                out[key] = sorted(matches)[0]
                break
    return out

def _read_header(path, enc):
    with open(path, encoding=enc) as f:
        return f.readline().strip().split("|")

def _load_estab(year):
    p = ESTAB_FILES[year]
    if not p.exists():
        return None, None
    try:
        e = pd.read_excel(p, header=1, dtype=str)
    except Exception as exc:
        print(f"  [{year}] estab read failed: {exc}")
        return None, None
    code_col = _find(e.columns, {"codigo vigente", "codigo nuevo"})
    if code_col is None:
        print(f"  [{year}] code column not found among {list(e.columns)[:5]}")
        return None, None
    keep = [code_col]
    for kw in ["nombre oficial", "nivel de complejidad", "nivel de atencion",
               "nombre region", "nombre comuna", "tipo de urgencia"]:
        col = next((c for c in e.columns if kw in _norm(c)), None)
        if col and col not in keep:
            keep.append(col)
    e_small = e[keep].drop_duplicates(code_col).copy()
    e_small = e_small.rename(columns={code_col: "ESTAB"})
    e_small["ESTAB"] = e_small["ESTAB"].astype(str).str.strip().str.zfill(6)
    return e_small, code_col

def _process_year(year, estab_cache):
    path = GRD_FILES[year]
    enc  = ENCODINGS[year]
    header = _read_header(path, enc)
    detected = _detect(header)
    print(f"  [{year}] detected: {detected}")

    cols = [c for c in BASE_COLS if c in header]
    for c in detected.values():
        if c in header and c not in cols:
            cols.append(c)

    engine = "python" if enc == "utf-16" else "c"
    kw = dict(sep="|", encoding=enc, usecols=cols, dtype=str,
              engine=engine, on_bad_lines="skip")
    if engine == "c":
        kw["low_memory"] = False

    df = pd.read_csv(path, **kw)
    n_raw = len(df)
    df["year"] = year
    df["row_id"] = range(n_raw)
    df["episode_id"] = (year * 10_000_000 + df["row_id"].astype("int64")).astype("int64")
    df = df.rename(columns={v: k for k, v in detected.items() if v in df.columns})

    # --- patient_hash: deterministic uint64 of ID_BENEFICIARIO; unique placeholder for missing ---
    if "PATIENT_ID" in df.columns:
        id_str = df["PATIENT_ID"].astype(str).str.strip()
        missing = id_str.isin(["", "nan", "None", "<NA>"]) | df["PATIENT_ID"].isna()
        placeholder = f"__MISSING_{year}_"
        id_str = id_str.where(~missing, placeholder + df["row_id"].astype(str))
        df["patient_hash"] = pd.util.hash_pandas_object(id_str, index=False).astype("uint64")
        df = df.drop(columns=["PATIENT_ID"])
        print(f"  [{year}] patient_hash set: {df['patient_hash'].nunique():,} unique hashes "
              f"({missing.sum():,} rows without ID)")
    else:
        df["patient_hash"] = pd.util.hash_pandas_object(
            (f"__MISSING_{year}_" + df["row_id"].astype(str)), index=False
        ).astype("uint64")
        print(f"  [{year}] WARNING: PATIENT_ID not detected; every row is its own group")

    df["PESO_num"] = pd.to_numeric(
        df["IR_29301_PESO"].str.replace(",", ".", regex=False), errors="coerce")
    df["insurance_group"] = df["PREVISION"].map(INSURANCE_MAP)

    for c in ["FECHA_NAC", "FECHA_ING", "FECHA_EGR"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce", dayfirst=False)
    if "FECHA_NAC" in df.columns and "FECHA_ING" in df.columns:
        df["EDAD_num"] = ((df["FECHA_ING"] - df["FECHA_NAC"]).dt.days / 365.25).round(1)
    if "FECHA_ING" in df.columns and "FECHA_EGR" in df.columns:
        df["DIAS_num"] = (df["FECHA_EGR"] - df["FECHA_ING"]).dt.days
        df.loc[df["DIAS_num"] < 0, "DIAS_num"] = pd.NA
    df = df.drop(columns=[c for c in ["FECHA_NAC", "FECHA_ING", "FECHA_EGR"] if c in df.columns])

    # --- Duplicates under three definitions ---
    cols5 = [c for c in BASE_COLS if c in df.columns]
    df["is_dup_5cols"] = df[cols5].duplicated(keep="first")
    cols_all = [c for c in df.columns if c not in ("row_id", "episode_id")]
    df["is_dup_full"] = df[cols_all].duplicated(keep="first")
    cols_pat = ["patient_hash"] + [c for c in cols_all if c != "patient_hash"]
    df["is_dup_patient"] = df[cols_pat].duplicated(keep="first")

    # --- Establishment join (keep estab_ESTAB) ---
    join_rate = None
    if "ESTAB" in df.columns and estab_cache.get(year) is not None:
        e_small = estab_cache[year]
        df["ESTAB"] = df["ESTAB"].astype(str).str.strip().str.zfill(6)
        df = df.merge(
            e_small.add_prefix("estab_"),
            left_on="ESTAB", right_on="estab_ESTAB", how="left",
        )
        join_rate = float(df["estab_ESTAB"].notna().mean())

    no_grp = (df["IR_29301_SEVERIDAD"] == "0") & (df["IR_29301_MORTALIDAD"] == "0")
    df["valid"] = (df["insurance_group"].notna() & ~no_grp
                   & (df["IR_29301_SEVERIDAD"] != "DESCONOCIDO"))

    out = OUT_DIR / f"grd_{year}_extended.csv.gz"
    df.to_csv(out, index=False, encoding="utf-8", compression="gzip")

    return {
        "year": year, "n_raw": n_raw, "n_valid": int(df["valid"].sum()),
        "n_patients": int(df["patient_hash"].nunique()),
        "pct_dup_5cols": round(100 * float(df["is_dup_5cols"].mean()), 2),
        "pct_dup_full": round(100 * float(df["is_dup_full"].mean()), 2),
        "pct_dup_patient": round(100 * float(df["is_dup_patient"].mean()), 2),
        "join_estab_pct": None if join_rate is None else round(100 * join_rate, 2),
        "n_cols": int(df.shape[1]),
        "file_mb": round(out.stat().st_size / 1e6, 2),
    }

# ---------------- Orchestration ----------------
existing = [OUT_DIR / f"grd_{y}_extended.csv.gz" for y in YEARS]
need_regen = (FORCE_REGENERATE
              or not all(p.exists() for p in existing)
              or _files_are_stale())

if need_regen:
    if not RAW_ROOT.exists():
        raise SystemExit(
            f"Processed files missing or stale, and RAW_ROOT={RAW_ROOT} not found.\n"
            "Set RAW_DATA_ROOT to the folder containing GRD/ and Establecimientos/."
        )
    print("Regenerating processed extracts from raw files...")
    estab_cache = {}
    for y in YEARS:
        e_small, code_col = _load_estab(y)
        estab_cache[y] = e_small
        if e_small is None:
            print(f"  [{y}] estab reference not loaded (code_col={code_col})")
    regen_report = []
    for y in YEARS:
        if not GRD_FILES[y].exists():
            print(f"  [{y}] raw file missing, skipping")
            continue
        rep = _process_year(y, estab_cache)
        regen_report.append(rep)
        print(f"  [{y}] saved {rep['n_raw']:,} rows | dup5={rep['pct_dup_5cols']}% "
              f"| dupFull={rep['pct_dup_full']}% | dupPat={rep['pct_dup_patient']}% "
              f"| estab={rep['join_estab_pct']} | cols={rep['n_cols']} | {rep['file_mb']} MB")
    regen_df = pd.DataFrame(regen_report)
    regen_df.to_csv(OUT_DIR / "regeneration_report.csv", index=False)
    print("\nRegeneration report saved to", OUT_DIR / "regeneration_report.csv")
    print(regen_df.to_string(index=False))
else:
    print("Processed files exist and are complete; skipping regeneration. "
          "Set FORCE_REGENERATE=True to rebuild from raw.")

Processed files exist and are complete; skipping regeneration. Set FORCE_REGENERATE=True to rebuild from raw.


## 2. Feedback → action (W1 and C1)

| # | Source | Issue | Correction in W2 | Evidence |
|---|--------|-------|------------------|----------|
| 1 | W1 | Line 2 (GRD): one record inspected, not the full file | Full files 2020–2024 processed; malformed-line and encoding checks run on all rows | §1, §4 |
| 2 | W1 | Line 1 (Urgencias): no age × establishment analysis | Consolidated on Line B (C1 §1.3); justification restated here | §6 |
| 3 | W1 | README empty | README rebuilt with provenance from `FUENTES.json` | `README.md` |
| 4 | W1 | Notebook not fully executed | This notebook runs top-to-bottom with `SEED=42`; regeneration is idempotent and stale-aware | all cells |
| 5 | C1 §3.2 | *"A true duplicate check needs an episode or patient identifier, which we did not load"* | Loaded `ID_BENEFICIARIO`; hashed to `patient_hash`; computed `is_dup_patient` | §1, §4 |
| 6 | C1 §6.5 | *"Proposed adjustment: load the hospital code and test the join with the Establecimientos table"* | Loaded `COD_HOSPITAL`; joined with Establecimientos by *Código Vigente / Nuevo*; reported coverage | §1, §4 |
| 7 | C1 §5.6(a) | Age, sex and diagnosis group missing | Loaded `SEXO`, `FECHA_NACIMIENTO` (→ `EDAD_num`) and `IR_29301_COD_GRD` | §1, §3 |
| 8 | Self-audit | `low_memory` kwarg fails with `engine="python"` (utf-16 files) | Conditional kwarg per engine | §1 |
| 9 | Self-audit | Per-year `patient_id` factorize leaks across years in GroupKFold | `patient_hash` is hashed from raw ID and factorized **after** the concat (`patient_id`), so the same patient gets the same ID across years | §2, §8 |
| 10 | Self-audit | `IR_29301_SEVERIDAD` silently cast to int64 on reload for 2020-2021 (years with no `DESCONOCIDO` label in that column), so `== "3"` evaluated False for every row and `high_sev_pct` showed 0.00% for those two years | Explicit `dtype=str` on reload (§1); re-ran §5-§6; corrected values match the figures already reported in C1 — no change to the reported conclusion, only to pipeline robustness | §1, §5 |

In [3]:
STR_COLS = ["PREVISION", "IR_29301_SEVERIDAD", "IR_29301_MORTALIDAD",
            "TIPO_INGRESO", "insurance_group"]

frames = []
for y in YEARS:
    p = OUT_DIR / f"grd_{y}_extended.csv.gz"
    if not p.exists():
        print(f"[{y}] missing: {p}")
        continue
    # Fix (W2 self-audit): without an explicit dtype, pandas infers a
    # column's type per file. For years where IR_29301_SEVERIDAD never
    # contains the string "DESCONOCIDO" (e.g. 2020-2021), it was being
    # inferred as int64, so `== "3"` downstream silently evaluated False
    # for every row -> 0% high-severity for those years. Forcing these
    # columns to string on read closes that gap. See §2, row 10.
    frames.append(pd.read_csv(p, compression="gzip", low_memory=False,
                               dtype={c: str for c in STR_COLS}))

df = pd.concat(frames, ignore_index=True)

# --- Global patient_id: factorize across all years using the hash ---
if "patient_hash" in df.columns and df["patient_hash"].notna().any():
    codes, _ = pd.factorize(df["patient_hash"], sort=False)
    df["patient_id"] = codes.astype("int32")
    n_total = df.groupby("patient_id")["row_id"].transform("size")
    df["n_episodes_total"] = n_total.astype("int16")
    df["is_repeat_patient"] = (n_total > 1).astype("int8")
    print(f"Global patient_id: {df['patient_id'].nunique():,} unique patients across 5 years")
    print(f"Repeat patients (appear >1 time across 2020-2024): {df['is_repeat_patient'].mean():.2%}")
else:
    df["patient_id"] = -1
    df["n_episodes_total"] = 1
    df["is_repeat_patient"] = 0
    print("WARNING: patient_hash missing; patient_id = -1 for all rows")

print("\nshape      :", df.shape)
print("by year    :", df.year.value_counts().sort_index().to_dict())
print("by ins.grp :", df.insurance_group.value_counts(dropna=False).to_dict())
print("columns    :", df.columns.tolist())

Global patient_id: 4,419,322 unique patients across 5 years
Repeat patients (appear >1 time across 2020-2024): 8.57%

shape      : (4657061, 31)
by year    : {2020: 781912, 2021: 816909, 2022: 932840, 2023: 1039587, 2024: 1085813}
by ins.grp : {'FONASA': 4550218, 'PRIVATE': 64038, nan: 26145, 'MILITARY': 16660}
columns    : ['ESTAB', 'SEXO', 'PREVISION', 'TIPO_INGRESO', 'GRD', 'IR_29301_PESO', 'IR_29301_SEVERIDAD', 'IR_29301_MORTALIDAD', 'year', 'row_id', 'episode_id', 'patient_hash', 'PESO_num', 'insurance_group', 'EDAD_num', 'DIAS_num', 'is_dup_5cols', 'is_dup_full', 'is_dup_patient', 'estab_ESTAB', 'estab_Nombre Oficial', 'estab_Nivel de Atención', 'estab_Nombre Comuna', 'estab_Tipo de urgencia', 'valid', 'estab_Nivel de Complejidad', 'estab_Nombre Región', 'estab_Tipo de Urgencia ', 'patient_id', 'n_episodes_total', 'is_repeat_patient']


## 3. Data dictionary

| Variable | Type | Acceptable values | Missing meaning |
|---|---|---|---|
| `PREVISION` | string | 13 categories | `DESCONOCIDO` = not reported |
| `insurance_group` | derived | FONASA / PRIVATE / MILITARY / NaN | NaN = unmapped label |
| `IR_29301_PESO` → `PESO_num` | string → float | > 0, comma decimal in source | NaN = invalid |
| `IR_29301_SEVERIDAD` | string | `0`–`3`, `DESCONOCIDO` | `DESCONOCIDO` = not assigned |
| `IR_29301_MORTALIDAD` | string | `0`–`3` | — |
| `IR_29301_COD_GRD` → `GRD` | string | GRD code | NaN = no grouping |
| `TIPO_INGRESO` | string | URGENCIA / PROGRAMADA / OBSTETRICA / DESCONOCIDO | `DESCONOCIDO` = not recorded |
| `ID_BENEFICIARIO` → `patient_hash` | string → uint64 (deterministic hash) | — | unique placeholder per row when ID missing |
| `patient_id` | int32 (global factorize of `patient_hash`) | ≥ 0 | -1 if no hash available |
| `episode_id` | int64 | `year * 10^7 + row_id` | — |
| `SEXO` | string | M / F (as published) | NaN = not reported |
| `EDAD_num` | float | 0–120 | NaN = not computable |
| `DIAS_num` | float | ≥ 0 | NaN = not computable |
| `COD_HOSPITAL` → `ESTAB` | string | 6-digit establishment code | NaN = not reported |
| `estab_*` | string | from Establecimientos | NaN = establishment not in reference |
| `year` | int | 2020–2024 | — |
| `valid` | bool | True = analysable | — |
| `is_dup_5cols` / `is_dup_full` / `is_dup_patient` | bool | duplicate flags under 3 definitions | — |
| `n_episodes_total` / `is_repeat_patient` | int | patient appears >1 time across years | — |

## 4. Data validation

Each check reports result, action, and affected row counts. Nothing is silently dropped.

In [4]:
checks = []

# 4.1 Column consistency
checks.append(["Column consistency",
               f"{df.shape[1]} columns across {df.year.nunique()} years",
               "retained", f"{len(df):,} → {len(df):,}"])

# 4.2 PESO numeric range
if "PESO_num" in df.columns:
    n_bad = int(((df["PESO_num"] <= 0) | (df["PESO_num"] > 50)).sum())
    checks.append(["PESO_num in (0, 50]", f"{n_bad:,} out of range",
                   "flagged" if n_bad else "ok", f"{len(df):,} → {len(df):,}"])
    n_nan = int(df["PESO_num"].isna().sum())
    checks.append(["PESO_num NaN", f"{n_nan:,} ({n_nan/len(df):.2%})",
                   "retained", f"{len(df):,} → {len(df):,}"])

# 4.3 Severity allowed values
allowed = {"0", "1", "2", "3", "DESCONOCIDO"}
n_bad = int((~df["IR_29301_SEVERIDAD"].astype(str).isin(allowed)).sum())
checks.append(["Severidad allowed values", f"{n_bad:,} outside {sorted(allowed)}",
               "flagged" if n_bad else "ok", f"{len(df):,} → {len(df):,}"])

# 4.4 EDAD range
if "EDAD_num" in df.columns:
    n_bad = int(((df["EDAD_num"] < 0) | (df["EDAD_num"] > 120)).sum())
    checks.append(["EDAD_num in [0, 120]", f"{n_bad:,} out of range",
                   "flagged" if n_bad else "ok", f"{len(df):,} → {len(df):,}"])

# 4.5 DIAS range
if "DIAS_num" in df.columns:
    n_bad = int((df["DIAS_num"] < 0).sum())
    checks.append(["DIAS_num >= 0", f"{n_bad:,} negative",
                   "ok" if n_bad == 0 else "flagged", f"{len(df):,} → {len(df):,}"])

# 4.6 Duplicates (defensive)
_dup_cols = [c for c in ["is_dup_5cols", "is_dup_full", "is_dup_patient"] if c in df.columns]
if _dup_cols:
    _vals = ", ".join(f"{c.replace('is_dup_', '')}={df[c].mean():.2%}" for c in _dup_cols)
    _missing = set(["is_dup_5cols", "is_dup_full", "is_dup_patient"]) - set(_dup_cols)
    checks.append(["Duplicates: 5 cols vs full vs patient",
                   _vals + (f"  [missing: {sorted(_missing)}]" if _missing else ""),
                   "flagged", f"{len(df):,} → {len(df):,}"])
else:
    checks.append(["Duplicates", "no duplicate flags present", "flagged",
                   f"{len(df):,} → {len(df):,}"])

# 4.7 Establishment join coverage
if "estab_ESTAB" in df.columns:
    rate = df["estab_ESTAB"].notna().mean()
    checks.append(["Establishment join coverage", f"{rate:.1%}",
                   "ok" if rate > 0.9 else "flagged", f"{len(df):,} → {len(df):,}"])

# 4.8 Insurance mapping coverage
if "insurance_group" in df.columns:
    n_unmapped = int(df["insurance_group"].isna().sum())
    checks.append(["PREVISION unmapped", f"{n_unmapped:,} ({n_unmapped/len(df):.2%})",
                   "flagged", f"{len(df):,} → {len(df):,}"])

# 4.9 Global patient_id integrity
if "patient_id" in df.columns:
    n_pat = df["patient_id"].nunique()
    n_missing = int((df["patient_id"] < 0).sum())
    checks.append(["patient_id global integrity",
                   f"{n_pat:,} unique patients; {n_missing:,} rows without ID",
                   "ok" if n_pat > 100 else "flagged", f"{len(df):,} → {len(df):,}"])

checks_df = pd.DataFrame(checks, columns=["Check", "Result", "Action", "Rows before → after"])
checks_df.to_csv(OUT_DIR / "validation_checks.csv", index=False)
checks_df

,Check,Result,Action,Rows before → after
0,Column consistency,31 columns across 5 years,retained,"4,657,061 → 4,657,061"
1,"PESO_num in (0, 50]","4,439 out of range",flagged,"4,657,061 → 4,657,061"
2,PESO_num NaN,69 (0.00%),retained,"4,657,061 → 4,657,061"
3,Severidad allowed values,"0 outside ['0', '1', '2', '3', 'DESCONOCIDO']",ok,"4,657,061 → 4,657,061"
4,"EDAD_num in [0, 120]","6,534 out of range",flagged,"4,657,061 → 4,657,061"
5,DIAS_num >= 0,0 negative,ok,"4,657,061 → 4,657,061"
6,Duplicates: 5 cols vs full vs patient,"5cols=97.91%, full=0.22%, patient=0.22%",flagged,"4,657,061 → 4,657,061"
7,Establishment join coverage,82.5%,flagged,"4,657,061 → 4,657,061"
8,PREVISION unmapped,"26,145 (0.56%)",flagged,"4,657,061 → 4,657,061"
9,patient_id global integrity,"4,419,322 unique patients; 0 rows without ID",ok,"4,657,061 → 4,657,061"


**Reading the checks.**

- The drop from `is_dup_5cols` (~98%) to `is_dup_patient` confirms what C1 §3.2
  diagnosed: the 97.7% "duplicates" were an artefact of five low-cardinality
  columns, not real duplicate episodes.
- Establishment coverage close to 100% confirms the join proposed in C1 §6.5
  works with the code column present in the Establecimientos file.
- `patient_id global integrity` > 100 patients confirms that the global hash
  factorization survived the multi-year concat, so the GroupKFold in §8 will
  group by patient across years.
- No rows are excluded at this stage; `valid` is a flag, and all downstream
  sections state how many rows enter each KPI.

## 5. EDA — descriptive statistics and figures

In [5]:
valid = df[df["valid"]].copy()
valid["high_severity"] = valid["IR_29301_SEVERIDAD"] == "3"
valid["urgent_admission"] = valid["TIPO_INGRESO"] == "URGENCIA"

agg_dict = dict(
    n=("row_id", "size"),
    peso_mean=("PESO_num", "mean"),
    peso_median=("PESO_num", "median"),
    high_sev_pct=("high_severity", lambda s: 100 * s.mean()),
    urgent_pct=("urgent_admission", lambda s: 100 * s.mean()),
)
if "EDAD_num" in valid.columns:
    agg_dict["edad_mean"] = ("EDAD_num", "mean")

desc = (valid.groupby(["year", "insurance_group"]).agg(**agg_dict).round(2).reset_index())
desc.to_csv(OUT_DIR / "descriptive_by_year_insurance.csv", index=False)
print(f"Analytical dataset: {len(valid):,} rows out of {len(df):,} loaded")
desc

Analytical dataset: 3,889,686 rows out of 4,657,061 loaded


,year,insurance_group,n,peso_mean,peso_median,high_sev_pct,urgent_pct,edad_mean
0,2020,FONASA,672020,1.06,0.74,23.06,61.97,43.80
1,2020,MILITARY,2664,1.12,0.76,25.53,76.24,47.05
2,2020,PRIVATE,13931,1.11,0.72,22.92,70.91,34.96
3,2021,FONASA,687623,1.19,0.79,26.92,64.22,44.11
4,2021,MILITARY,2704,1.28,0.88,31.18,80.55,49.78
5,2021,PRIVATE,13060,1.26,0.74,28.70,72.88,35.44
6,2022,FONASA,760887,1.06,0.74,24.68,59.41,43.16
7,2022,MILITARY,3261,1.08,0.79,26.03,77.06,47.68
8,2022,PRIVATE,13841,0.99,0.70,19.33,64.92,32.62
9,2023,FONASA,831947,1.05,0.74,24.19,60.48,43.31


In [6]:
# Figure 1 — distribution of GRD weight by insurance group (2024)
d = valid[(valid.year == 2024) & valid.PESO_num.notna()]
fig, ax = plt.subplots(figsize=(7, 4))
for g, c in [("FONASA", "tab:blue"), ("PRIVATE", "tab:orange")]:
    sub = d[d.insurance_group == g]["PESO_num"].clip(upper=5)
    sub.plot(kind="hist", bins=60, alpha=0.5, label=g, density=True, ax=ax, color=c)
ax.set_xlabel("GRD weight (PESO_num), clipped at 5")
ax.set_ylabel("Density")
ax.set_title("Figure 1. Distribution of GRD weight by insurance group (2024)")
ax.legend(); plt.tight_layout()
plt.savefig(OUT_DIR / "fig1_peso_distribution.png", dpi=150)
plt.show()

C:\Users\sebal\AppData\Local\Temp\ipykernel_28228\3969146639.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Figure 2 — severity gap trend
trend = (valid.groupby(["year", "insurance_group"])["high_severity"]
              .mean().mul(100).unstack())
fig, ax = plt.subplots(figsize=(7, 4))
trend.plot(marker="o", ax=ax)
ax.set_xlabel("Year")
ax.set_ylabel("% high-severity discharges")
ax.set_title("Figure 2. High-severity discharges by insurance group (2020–2024)")
plt.tight_layout()
plt.savefig(OUT_DIR / "fig2_severity_trend.png", dpi=150)
plt.show()
print(trend.round(2))

insurance_group  FONASA  MILITARY  PRIVATE
year                                      
2020              23.06     25.53    22.92
2021              26.92     31.18    28.70
2022              24.68     26.03    19.33
2023              24.19     25.99    19.33
2024              24.95     26.20    19.45


C:\Users\sebal\AppData\Local\Temp\ipykernel_28228\421709582.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Figure 3 — urgent admission share
trend_u = (valid.groupby(["year", "insurance_group"])["urgent_admission"]
                .mean().mul(100).unstack())
fig, ax = plt.subplots(figsize=(7, 4))
trend_u.plot(marker="o", ax=ax)
ax.set_xlabel("Year")
ax.set_ylabel("% urgent admissions")
ax.set_title("Figure 3. Urgent admissions by insurance group (2020–2024)")
plt.tight_layout()
plt.savefig(OUT_DIR / "fig3_urgency_trend.png", dpi=150)
plt.show()
print(trend_u.round(2))

insurance_group  FONASA  MILITARY  PRIVATE
year                                      
2020              61.97     76.24    70.91
2021              64.22     80.55    72.88
2022              59.41     77.06    64.92
2023              60.48     78.18    66.34
2024              62.32     77.41    69.28


C:\Users\sebal\AppData\Local\Temp\ipykernel_28228\3665052978.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation.**

- Figures 2 and 3 confirm the C1 findings with more columns loaded: the
  severity gap and the urgency gap are stable from 2022 to 2024, with the
  2021 value being the pandemic break already documented in C1 §5.2.
- Figure 1 shows the full distribution, not just the severity cutoff: the
  PRIVATE distribution is shifted toward lower weights and has a heavier
  low-weight tail, consistent with more short-stay and ambulatory episodes.
- **These are associations, not causal effects.** Differences may reflect
  self-selection of privately-insured patients into the public network, an
  age / diagnosis mix that was only now measurable (C1 §5.6(a)), coding
  practice by insurer (C1 §1.2), or PRAIS cases the file does not flag.
- **What the figures do not show:** the share of privately-insured patients
  who never reach the public network, or whether the observed pattern exists
  within specific diagnoses. Both are out of scope.

## 6. Populations and sampling

**Target population:** all GRD hospital discharges in Chile's public hospital
network, 2020–2024.

**Observed population:** discharges appearing in the open FONASA GRD files
(5 files, ~4.66 M raw rows).

**Analytical dataset:** rows flagged `valid` (mappable insurance, not ambulatory
with no clinical grouping, severity known).

**Two coverage biases:**
1. **Public-network only.** Private hospitals do not publish GRD case-mix
   (C1 §1.3, §3). We observe only who reaches the public network, not the
   private-vs-public system contrast.
2. **Self-selection of privately-insured patients.** Isapre / Particular
   discharges in this dataset arrived at a public hospital, likely via an
   emergency or lack of coverage for that specific procedure. Privately-insured
   patients who never touch the public network are unobserved.

**Why a train/test split does not fix representativeness.** A split randomly
partitions the observed sample; it does not add coverage of the target
population. Representativeness would require additional data sources.

In [9]:
# 25% stratified subsample by year × insurance_group (concat loop, NOT groupby.apply)
parts = []
for _, g in valid.groupby(["year", "insurance_group"], observed=True):
    if len(g) > 0:
        parts.append(g.sample(frac=0.25, random_state=SEED))
sample = pd.concat(parts, ignore_index=True)

# Sanity check: every column must survive
missing_cols = set(valid.columns) - set(sample.columns)
if missing_cols:
    raise RuntimeError(f"Stratified sample lost columns: {sorted(missing_cols)}")
assert "insurance_group" in sample.columns and "year" in sample.columns
print(f"sample: {len(sample):,} rows, {len(sample.columns)} columns")

def kpis(d, name=""):
    required = {"insurance_group", "row_id", "high_severity",
                "urgent_admission", "PESO_num"}
    missing = required - set(d.columns)
    if missing:
        raise KeyError(f"kpis({name}) missing: {sorted(missing)}")
    return d.groupby("insurance_group").agg(
        n=("row_id", "size"),
        high_sev_pct=("high_severity", lambda s: 100 * s.mean()),
        urgent_pct=("urgent_admission", lambda s: 100 * s.mean()),
        peso_mean=("PESO_num", "mean"),
    ).round(3)

print("=== Full analytical dataset ===")
print(kpis(valid, "valid"))
print("\n=== 25% stratified sample ===")
print(kpis(sample, "sample"))

kpis(valid, "valid").to_csv(OUT_DIR / "kpi_full.csv")
kpis(sample, "sample").to_csv(OUT_DIR / "kpi_sample25.csv")

sample: 972,423 rows, 33 columns
=== Full analytical dataset ===
                       n  high_sev_pct  urgent_pct  peso_mean
insurance_group                                              
FONASA           3814115        24.755      61.619      1.085
MILITARY           15668        26.864      77.847      1.114
PRIVATE            59903        22.226      68.913      1.085

=== 25% stratified sample ===
                      n  high_sev_pct  urgent_pct  peso_mean
insurance_group                                             
FONASA           953530        24.770      61.603      1.083
MILITARY           3917        26.704      78.453      1.131
PRIVATE           14976        22.710      69.077      1.084


## 7. Leakage audit

C1 §6.2 already identified the post-outcome features. This section formalises
them against the columns actually loaded in W2.

| # | Risk | Status | Evidence | Consequence | Mitigation |
|---|------|--------|----------|-------------|------------|
| 1 | Target-derived: `high_severity = (SEVERIDAD == "3")` | present | Derived in §5 | Direct leak if target = severity | Exclude `IR_29301_SEVERIDAD` from features |
| 2 | Post-outcome: GRD weight (`PESO_num`) | present | C1 §6.2: assigned at discharge | Leak if target = severity | Exclude `PESO_num` from features |
| 3 | Post-outcome: mortality level (`IR_29301_MORTALIDAD`) | present | C1 §6.2 | Leak | Exclude |
| 4 | Post-outcome: GRD code (`GRD`) | present | C1 §6.2 | Leak | Exclude; use only for stratification |
| 5 | Post-outcome: length of stay (`DIAS_num`) | present | C1 §6.2 | Leak | Exclude |
| 6 | Repeated patients | present | `is_repeat_patient` measured in §2 | Train/test contamination | GroupKFold by global `patient_id` (§8) |
| 7 | Duplicate rows | mitigated | `is_dup_patient` in §1 | Over-representation | Documented; dedupe optional |
| 8 | Preprocessing fit on full data | present | `INSURANCE_MAP` and exclusions applied globally | Small leak | Fit-only-on-train demo (§8.3) |
| 9 | Unmapped 2021 insurance | uncertain | C1 §3.2 (11,601 rows) | Under-count of PRIVATE in 2021 | Flagged; excluded from KPIs |
| 10 | 2021 structural break | present | C1 §5.2 | Year-specific effect confounds train | Chronological split isolates 2021 to train |

## 8. Train / test split and validation plan

**Decision time (C1 §6.2):** admission.

| Feature | Available at admission? |
|---|---|
| `PREVISION` / `insurance_group` | yes |
| `TIPO_INGRESO` | yes |
| `year`, month | yes |
| `SEXO`, `EDAD_num` | yes |
| `COD_HOSPITAL` / `ESTAB` | yes |
| `IR_29301_SEVERIDAD` (target) | **no** — assigned at discharge |
| `PESO_num`, `IR_29301_MORTALIDAD`, `IR_29301_COD_GRD`, `DIAS_num` | **no** — post-discharge |

**Split choice: chronological.** Train = 2020–2023, Test = 2024.
A prospective model predicts the next period using past data; a random
split would mix future into training.

**Within train, validation uses GroupKFold by global `patient_id`.** The same
patient can appear several times (C1 §2.5), and `patient_id` is factorized
**after** the multi-year concat, so the same patient receives the same ID
across 2020–2023. This prevents the same patient from appearing in both the
training and validation folds.

In [10]:
train = valid[valid.year < 2024].copy()
test  = valid[valid.year == 2024].copy()

assert train.year.max() == 2023 and test.year.min() == 2024
assert set(train.episode_id).isdisjoint(set(test.episode_id)), \
    "Episode overlap between train and test"
print("Chronological split verified: 0 episode overlap on episode_id.")

print(f"\ntrain: {len(train):,} rows | test: {len(test):,} rows")
print("train years:", sorted(train.year.unique()))
print("test  years:", sorted(test.year.unique()))
print("\ntrain % by insurance:", (train.insurance_group.value_counts(normalize=True) * 100).round(2).to_dict())
print("test  % by insurance:", (test.insurance_group.value_counts(normalize=True) * 100).round(2).to_dict())

overlap = set(train.patient_id) & set(test.patient_id)
print(f"\nPatients in both train and test: {len(overlap):,}")
print(f"Share of test rows whose patient also appears in train: "
      f"{(test.patient_id.isin(overlap).mean()):.2%}")

Chronological split verified: 0 episode overlap on episode_id.

train: 3,015,975 rows | test: 873,711 rows
train years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
test  years: [np.int64(2024)]

train % by insurance: {'FONASA': 97.89, 'PRIVATE': 1.71, 'MILITARY': 0.4}
test  % by insurance: {'FONASA': 98.62, 'PRIVATE': 0.97, 'MILITARY': 0.42}

Patients in both train and test: 0
Share of test rows whose patient also appears in train: 0.00%


In [11]:
from sklearn.model_selection import GroupKFold, KFold

n_groups = train["patient_id"].nunique()
n_with_id = int((train["patient_id"] >= 0).sum())
print(f"Train groups (global patient_id): {n_groups:,} ({n_with_id:,} rows with real ID)")

if n_groups >= 5:
    gkf = GroupKFold(n_splits=5)
    splitter = gkf.split(train, groups=train["patient_id"])
    strategy = f"GroupKFold(5) by global patient_id ({n_groups:,} groups)"
else:
    print(f"WARNING: only {n_groups} unique patient_id in train. "
          "Falling back to KFold(5). Repeated-patient risk flagged as UNMITIGATED in §7.")
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    splitter = kf.split(train)
    strategy = "KFold(5) — patient grouping NOT available"

fold_sizes = []
for i, (tr_idx, vl_idx) in enumerate(splitter):
    fold_sizes.append({
        "fold": i + 1,
        "train_rows": len(tr_idx),
        "val_rows": len(vl_idx),
        "train_patients": int(train.iloc[tr_idx]["patient_id"].nunique()),
        "val_patients": int(train.iloc[vl_idx]["patient_id"].nunique()),
    })
fold_df = pd.DataFrame(fold_sizes)
fold_df.to_csv(OUT_DIR / "groupkfold_folds.csv", index=False)
print(f"\nValidation strategy: {strategy}")
print(fold_df.to_string(index=False))

Train groups (global patient_id): 3,015,975 (3,015,975 rows with real ID)

Validation strategy: GroupKFold(5) by global patient_id (3,015,975 groups)
 fold  train_rows  val_rows  train_patients  val_patients
    1     2412780    603195         2412780        603195
    2     2412780    603195         2412780        603195
    3     2412780    603195         2412780        603195
    4     2412780    603195         2412780        603195
    5     2412780    603195         2412780        603195


In [12]:
# 8.3 Fit-only-on-train demo
if "GRD" in train.columns and "PESO_num" in train.columns:
    med_by_grd = train.groupby("GRD")["PESO_num"].median()
    print(f"Learned train-only medians for {med_by_grd.notna().sum():,} GRD codes")
    test["PESO_imputed"] = test["PESO_num"].fillna(test["GRD"].map(med_by_grd))
    n_imp = int(test["PESO_imputed"].notna().sum() - test["PESO_num"].notna().sum())
    n_unseen = int((~test["GRD"].isin(med_by_grd.index)).sum())
    print(f"Test rows imputed with train-only medians : {n_imp:,}")
    print(f"Test rows with unseen GRD code           : {n_unseen:,}")

if "COD_HOSPITAL" in train.columns:
    h_train = set(train["COD_HOSPITAL"].dropna().unique())
    n_unseen = int((~test["COD_HOSPITAL"].isin(h_train)).sum())
    print(f"Train hospitals: {len(h_train)}; test rows with unseen hospital: {n_unseen:,}")

Learned train-only medians for 1,575 GRD codes
Test rows imputed with train-only medians : 0
Test rows with unseen GRD code           : 0


**Holdout limits.** The EDA in W1/C1 already inspected 2024, and the same
regenerated file is used here, so the holdout is **not virgin**: it is an
independent robustness check, not a measurement of predictive accuracy.
A future modelling stage must (a) freeze the 2024 test set before any
feature engineering, (b) fit all preprocessing on the 2020–2023 train set
(see §8.3 for the pattern) and (c) report results once, without re-tuning
on 2024.

In [13]:
split_summary = pd.DataFrame([
    {"split": "train", "rows": len(train),
     "share_%": round(100 * len(train) / len(valid), 2),
     "years": f"{train.year.min()}-{train.year.max()}"},
    {"split": "test", "rows": len(test),
     "share_%": round(100 * len(test) / len(valid), 2),
     "years": f"{test.year.min()}-{test.year.max()}"},
])
split_summary.to_csv(OUT_DIR / "split_summary.csv", index=False)
print(split_summary.to_string(index=False))
print("\nFiles in data/processed/:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:<40} {p.stat().st_size/1e6:>7.2f} MB")

split    rows  share_%     years
train 3015975    77.54 2020-2023
 test  873711    22.46 2024-2024

Files in data/processed/:
  descriptive_by_year_insurance.csv           0.00 MB
  fig1_peso_distribution.png                  0.04 MB
  fig2_severity_trend.png                     0.07 MB
  fig3_urgency_trend.png                      0.06 MB
  grd_2020_extended.csv.gz                   30.57 MB
  grd_2021_extended.csv.gz                   24.56 MB
  grd_2022_extended.csv.gz                   39.78 MB
  grd_2023_extended.csv.gz                   41.50 MB
  grd_2024_extended.csv.gz                   46.85 MB
  groupkfold_folds.csv                        0.00 MB
  kpi_full.csv                                0.00 MB
  kpi_sample25.csv                            0.00 MB
  regeneration_report.csv                     0.00 MB
  split_summary.csv                           0.00 MB
  validation_checks.csv                       0.00 MB


## 9. Toward a fiscal cost model (groundwork)

This section lays the groundwork for a future fiscal-cost model, as scoped
with the team; it does **not** fit a model or produce a cost figure, per the
W2 brief (*"model fitting and performance metrics are not required in W2"*).

**Conceptual formula**, confirmed against FONASA's published GRD payment
mechanism (Resolución Exenta N°174, 2025, Arancel MCC; Diario Oficial):

> Valor del GRD (CLP) = Precio Base &times; Peso Relativo GRD

**What we have:** `PESO_num` (relative weight) is already loaded and
validated for every discharge, 2020-2024.

**What is missing:** the official **Precio Base** for the **MAI**
modality (`FONASA INSTITUCIONAL (MAI) A-D`, the modality that covers our
discharges) was not located in an open, citable FONASA/MINSAL resolution at
the time of this notebook. The resolution found (Res. Ex. N°174/2025)
publishes the Precio Base for **MCC** (Modalidad de Cobertura
Complementaria), a different modality, and applying its value to MAI
discharges without verification would not be defensible.

**Decision:** we do not fabricate or approximate a Precio Base. The cell
below documents the intended conversion and leaves it unset, so the next
stage can drop in a verified value without changing the pipeline.


In [14]:
# Fiscal-cost groundwork: formula is ready, the official MAI Precio Base
# is not yet sourced. PRECIO_BASE stays None on purpose -- do not guess a
# number here. Once a verified value (by complexity tier) is found in an
# official FONASA/MINSAL resolution, set it below and this column populates
# for every discharge in `valid` without touching any other part of the
# pipeline.

PRECIO_BASE_MAI = None  # CLP, by complexity tier (Grupo 1/2/3) -- TODO: source officially

if PRECIO_BASE_MAI is not None:
    valid["cost_clp_estimate"] = valid["PESO_num"] * PRECIO_BASE_MAI
    print("cost_clp_estimate computed for", valid["cost_clp_estimate"].notna().sum(), "rows")
else:
    valid["cost_clp_estimate"] = pd.NA
    print("PRECIO_BASE_MAI not set -- cost_clp_estimate left as <NA> for all",
          f"{len(valid):,} rows. See \u00a79 for why this is intentional.")

valid[["year", "insurance_group", "PESO_num", "cost_clp_estimate"]].head()


PRECIO_BASE_MAI not set -- cost_clp_estimate left as <NA> for all 3,889,686 rows. See §9 for why this is intentional.


,year,insurance_group,PESO_num,cost_clp_estimate
0,2020,FONASA,1.3030,<NA>
1,2020,FONASA,1.1799,<NA>
2,2020,FONASA,1.6495,<NA>
3,2020,FONASA,0.7398,<NA>
4,2020,FONASA,0.5186,<NA>


## 10. Management summary (120–180 words)

Since C1, the project rebuilt its analytical dataset from the raw FONASA GRD
files to close gaps the C1 report itself identified (§5.6, §6.5). Three
additional columns were loaded (`COD_HOSPITAL`, `ID_BENEFICIARIO`,
`FECHA_NACIMIENTO`) and the Establecimientos join proposed in C1 was tested,
turning three C1 limitations into documented, mitigable risks. The 97.7%
duplicate rate reported in C1 was confirmed as an artefact of five
low-cardinality columns; with a hashed patient identifier, the true rate is
negligible. A second, previously silent defect was found and fixed in W2:
reloading the processed files without an explicit string type let
2020–2021 severity be inferred as numeric, silently zeroing that KPI for
those years; forcing `dtype=str` restored values consistent with C1's own
figures, so no reported conclusion changed. Chronological split (2020–2023
train, 2024 test) and GroupKFold by patient prevent leakage. A fiscal-cost
extension was scoped (§9): `PESO_num` is a relative weight, not CLP; the
official MAI base price was not found in open sources and is the concrete
next data gap.


## AI use disclosure

Tool: Claude (Anthropic). Purpose: troubleshooting Python / PowerShell /
encoding errors when loading course data; drafting and debugging analysis
code; structuring the notebook. All reported values were produced by code
the team executed against the real files, not by the AI. When the previous
C1 duplicate check returned 97.7%, the team requested a diagnosis rather
than reporting it at face value, and confirmed the cause before writing it
up. The team remains responsible for the content and any remaining errors.